# 🔌 Module 1.3 — Subagent Invocation and Context Passing

**Domain 1 · Agentic Architecture & Orchestration** (27% of the exam)
**Task 1.3 · Subagent Invocation and Context Passing** · ⏱️ ~50 minutes
**Source:** [claudecertificationguide.com/learn/1-agentic-architecture/1-3-subagent-invocation-context](https://claudecertificationguide.com/learn/1-agentic-architecture/1-3-subagent-invocation-context)

### A different kind of module — read this before the setup cell

Modules 1.1 and 1.2 taught you to **hand-build** agentic loops and multi-agent
coordination on top of the raw Messages API — valuable for understanding the
mechanics underneath everything. This module is about the **real, built-in
primitive** that does that job for you in production Claude Code: the
`Task`/`Agent` tool, configured through the **Claude Agent SDK**
(`claude-agent-sdk` on PyPI — the actual Python SDK behind Claude Code itself).

That means this notebook looks different from the first two:

- It uses `claude_agent_sdk`, not the raw `anthropic` package
- It's **async** (`async for message in query(...)`) — Jupyter supports
  top-level `await` natively, so this runs directly in a cell
- The subagent isolation and context-relay machinery from Module 1.2 is
  handled *for you* by the SDK/CLI. Your job shifts to **configuration**
  (the `allowed_tools` gate, scoped `AgentDefinition`s) and **prompting**
  (making sure attribution metadata survives the relay) — then verifying it
  actually worked by reading the real message stream.

Every field name and shape below was checked against the **actually
installed** `claude-agent-sdk` package rather than assumed from the module's
own (TypeScript-flavored) pseudocode — a couple of real, useful discrepancies
turned up, called out as they come up.

### 🎯 What you'll build

A coordinator that spawns two isolated, tool-scoped subagents (web search +
document analysis), forces attribution metadata to survive the relay between
them, verifies it in the real output, and spawns independent work in
parallel — then, deliberately, breaks the attribution step exactly the way
the exam describes.

### ✅ What you'll walk away knowing

1. Why `Agent` (or its legacy alias `Task`) in `allowed_tools` is a hard,
   binary gate for subagent spawning
2. The real `AgentDefinition` shape, verified against the installed SDK
3. Why attribution failures are a coordinator context-passing bug, not a
   synthesis problem
4. How to spawn independent subagents in parallel instead of sequentially
5. `fork_session` (branch) vs. `resume` (continue) — and why the SDK asks
   for both together

---

> **💳 + ⏱️ Heads up, twice over:** this module's real demos do genuine web
> searches through actual subagents, so each real call can take **15–90+
> seconds** and costs a little more than 1.1/1.2's short text completions —
> `ResultMessage.total_cost_usd` will tell you the exact number afterward,
> so nothing here is a guess. Real web results also aren't scripted, so a
> couple of cells check what happened and explain it rather than asserting
> a single fixed outcome.

## 🔧 Setup

```bash
pip install claude-agent-sdk
```

`claude-agent-sdk` **bundles its own Claude Code CLI binary** (verified by
reading the installed package's source — it looks for a bundled binary
before ever falling back to a system-wide `claude` on PATH), so there's no
separate Node/npm install needed. Same environment variable as before:

```bash
# Windows (PowerShell)
$env:ANTHROPIC_API_KEY = "sk-ant-..."

# macOS / Linux (bash/zsh)
export ANTHROPIC_API_KEY="sk-ant-..."
```


In [ ]:
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. Set it in your shell, then restart the "
        "kernel and run this cell again -- see the Setup section above."
    )

from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AgentDefinition,
    AssistantMessage,
    UserMessage,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    ToolResultBlock,
)

print("claude_agent_sdk imported. Ready.")


## 🔑 Key Concept: The Task/Agent Tool Is a Hard Gate

The `Task` tool is "the actual API mechanism that makes multi-agent
orchestration work in the Claude Agent SDK, not a naming convention you can
skip past." Current Claude Code renamed it to **`Agent`**; `Task` remains a
valid legacy alias. The exam guide keys its answer as "Task tool"; current
tool-use blocks in production emit `"Agent"`.

**The rule, stated as a binary gate, not a preference:**

> "The coordinator's `allowedTools` must include `'Task'` (or `'Agent'`)...
> Without it, the coordinator cannot invoke any subagent regardless of how
> they are defined."

- Either `Agent` is in the coordinator's `allowed_tools` list, or the
  coordinator has **zero** subagent-spawning capability — no partial credit.
- Fully-defined `AgentDefinition`s sitting in `options.agents` do nothing on
  their own if the gate isn't open.
- Current SDK behavior: `allowed_tools` acts as an auto-approve list;
  omitting `Agent` routes the spawn attempt through a permission callback
  instead, which denies it in any unattended/non-interactive run — different
  mechanism, same practical outcome as the exam's "impossible" framing.


## 🔑 Key Concept: `AgentDefinition`

| Component | Purpose |
|---|---|
| **description** | What the subagent does — the coordinator reads this to decide when to invoke it |
| **prompt** | Instructions the subagent follows — goal-oriented, not procedural |
| **tools** | The subagent's scoped tool access — e.g. a search subagent gets search tools only |

> **Verified against the installed SDK, not assumed:** the module's own
> example pseudocode calls this field `system_prompt`. The real, installed
> `claude_agent_sdk.AgentDefinition` dataclass names it **`prompt`**. It's
> also worth knowing the real dataclass has a genuine internal
> inconsistency: `AgentDefinition.disallowedTools` is camelCase, while the
> *coordinator's* own `ClaudeAgentOptions.disallowed_tools` is snake_case.
> Real SDKs have real seams like this — checking the installed package
> beats trusting any guide's pseudocode literally, this one included.
>
> One more real shape worth knowing up front: `ClaudeAgentOptions.agents` is
> a **`dict[str, AgentDefinition]`** keyed by subagent name — not a list of
> objects each carrying their own `name` field, as the module's own
> illustrative snippet shows it.


## 🔑 Key Concept: Context Passing — Three Rules

**Rule 1 — Pass complete findings, every time.** "If the synthesis subagent
needs web search results and document analysis output, the coordinator must
pass both — in full — in the synthesis subagent's prompt. Do not assume the
synthesis agent can 'look up' prior results. It cannot." Same isolation
principle as Module 1.2, still absolute here.

**Rule 2 — Separate content from metadata, structurally.** "When passing
research findings between agents, the data must include both the content
(the claim) and the metadata (source URL, document name, page number). If
you pass content without metadata, the downstream agent cannot attribute
claims to sources." A `Finding` needs both halves, always:

```json
{
  "findings": [
    {
      "claim": "Solar panel efficiency has increased 25% in the last decade",
      "source_url": "https://example.com/solar-report",
      "document_name": "Annual Solar Industry Report 2024",
      "page_number": 14,
      "confidence": "high",
      "retrieved_by": "web_search_agent"
    }
  ]
}
```

**Rule 3 — Write goal-oriented prompts, not procedural ones.** Tell a
subagent *what* to achieve and what quality bar to meet, not the exact steps
to follow. Goal-oriented prompts let a subagent adapt when it hits something
unexpected; procedural ones just get in its way.


## 🔑 Key Concept: Parallel Spawning

"When a coordinator needs to invoke multiple subagents for independent
tasks, it should emit multiple Task tool calls **in a single response**
rather than invoking them one at a time across separate turns."

Sequential spawning of genuinely independent work just adds latency for no
benefit. **Exam signal:** an answer mentioning "in a single response" or
"simultaneously" is flagging this exact pattern as correct.


## 🔑 Key Concept: `fork_session` vs. `resume`

| | `fork_session` | `resume` |
|---|---|---|
| **Does** | Branches into a new, independent copy from a shared baseline | Continues a specific named session, appending to its history |
| **Use when** | Comparing divergent approaches from the same starting point | Continuing the same line of investigation |
| **Results shared between branches?** | No | N/A — it's the same line |

> "`fork_session` is not the same as `--resume`." The exam tests this
> distinction explicitly: resume continues, fork branches.

**Verified real SDK shape:** in the installed `claude_agent_sdk`, fork is a
**modifier on** resume, not an alternative to it — you pass both fields on
`ClaudeAgentOptions` together:

```python
# Reference pattern (verified against the real dataclass fields; not run
# here, to keep this module's already-heavier API cost in check):
ClaudeAgentOptions(resume=session_id, fork_session=True)
```

`resume` names *which* session; `fork_session=True` says *branch* from it
instead of appending to it. Leave `fork_session` at its default (`False`)
and `resume` alone just continues normally.


## 🛠️ Build Exercise — Task 1: Coordinator with `Agent` in `allowed_tools`

**Objective:** a coordinator configuration whose `allowed_tools` explicitly
contains `"Agent"` — the hard gate from the concept above, with no subagents
defined yet.

**Why this matters:** the exam tests this as a binary requirement, not a
runtime nicety. Get this wrong and nothing else in this notebook can work,
no matter how well-designed the `AgentDefinition`s below are.


In [ ]:
coordinator_options = ClaudeAgentOptions(
    allowed_tools=["Agent"],  # <-- the hard gate. "Task" remains a valid legacy alias.
    agents={},                # Task 2 fills this in.
)

print("allowed_tools:", coordinator_options.allowed_tools)
print("agents:       ", coordinator_options.agents)


## 🛠️ Build Exercise — Task 2: Two Scoped Subagents

**Objective:** a web-search subagent and a document-analysis subagent, each
with a description, a goal-oriented prompt, and **scoped tool access**
matching its role.

**Why this matters:** the exam tests whether you actually restrict each
subagent's tools to its role, not just whether you can write a prompt. We
also bake attribution into each subagent's prompt right here — Rule 2 from
the context-passing concept, applied at the source, not patched on later.


In [ ]:
web_search_agent = AgentDefinition(
    description="Performs web searches for factual research on a given subtopic.",
    prompt=(
        "You are a research subagent. Given a subtopic and a research goal, "
        "search the web for factual findings. For EVERY finding, report the "
        "exact source URL alongside the claim -- never state a claim without "
        "its source URL right next to it."
    ),
    tools=["WebSearch"],
)

document_analysis_agent = AgentDefinition(
    description="Fetches one specific URL and extracts a detailed, attributed analysis from it.",
    prompt=(
        "You are a document-analysis subagent. Given a specific URL and a "
        "research goal, fetch that exact page and extract detailed findings "
        "from it. Report the exact URL and page title alongside every "
        "finding -- never state a claim without its source."
    ),
    tools=["WebFetch"],
)

coordinator_options = ClaudeAgentOptions(
    allowed_tools=["Agent"],
    agents={
        "web_search_agent": web_search_agent,
        "document_analysis_agent": document_analysis_agent,
    },
)

print("Configured subagents:", list(coordinator_options.agents.keys()))
print("web_search_agent tools:      ", web_search_agent.tools)
print("document_analysis_agent tools:", document_analysis_agent.tools)


## 🛠️ Build Exercise — Task 3: A Structured `Finding` Format

**Objective:** a type that carries content *and* metadata together, so
neither can accidentally travel without the other.

**Why this matters:** this is Rule 2 made concrete as a schema, not just a
sentence. Nothing here calls the API — it's the shape everything downstream
gets checked against.


In [ ]:
from dataclasses import dataclass, asdict
from typing import Optional


@dataclass
class Finding:
    claim: str
    source_url: str
    document_name: str
    page_number: Optional[int]
    confidence: str        # "high" | "medium" | "low"
    retrieved_by: str      # which subagent produced this


example = Finding(
    claim="Solar panel efficiency has increased significantly over the last decade",
    source_url="https://example.com/solar-report",
    document_name="Annual Solar Industry Report",
    page_number=14,
    confidence="high",
    retrieved_by="web_search_agent",
)
print(asdict(example))


## 🛠️ Build Exercise — Tasks 4 & 5: Real Run, With Verification

**Objective (Task 4):** give the coordinator a prompt that hands off between
subagents *with* attribution intact — then let the SDK's own built-in
`Agent` tool handle the actual spawning and context relay, which is the one
thing you *don't* have to hand-build this time.

**Objective (Task 5):** verify the final synthesized output actually carries
citations for its claims, rather than assuming it does.

This is one real `query()` call. It does genuine web research through two
real subagents, so expect it to take **30–90+ seconds**, and check the
printed cost at the end rather than guessing.


In [ ]:
COORDINATOR_PROMPT = (
    "Research 'solar panel efficiency trends' using your available subagents. "
    "First, use web_search_agent to find 2-3 relevant sources. Then pick the "
    "single most relevant result and use document_analysis_agent to fetch and "
    "analyze it in depth, passing it that exact URL. Once you have findings "
    "from both, write a short synthesized report (3-5 sentences) that "
    "includes an inline citation (the source URL) for every factual claim. "
    "Do not drop any source URLs along the way."
)

final_report_text = None
total_cost = None

async for message in query(prompt=COORDINATOR_PROMPT, options=coordinator_options):
    if isinstance(message, AssistantMessage):
        origin = " (inside a subagent)" if message.parent_tool_use_id else " (coordinator)"
        for block in message.content:
            if isinstance(block, TextBlock):
                print(f"[assistant{origin}] {block.text}")
            elif isinstance(block, ToolUseBlock):
                print(f"[assistant{origin} -> tool_use] {block.name}({block.input})")

    elif isinstance(message, UserMessage) and isinstance(message.content, list):
        for block in message.content:
            if isinstance(block, ToolResultBlock):
                preview = str(block.content)[:200]
                print(f"[tool_result for {block.tool_use_id}] {preview}")

    elif isinstance(message, ResultMessage):
        final_report_text = message.result
        total_cost = message.total_cost_usd
        print()
        print("=== Final synthesized report ===")
        print(message.result)
        print()
        print(f"Turns: {message.num_turns}", end="")
        if total_cost is not None:
            print(f"  |  Cost: ${total_cost:.4f}")
        else:
            print()


In [ ]:
import re


def extract_urls(text: str) -> list:
    """Pull URLs out of free text, trimming trailing punctuation a sentence
    might leave attached (a closing paren, a period, ...) so
    "(https://x.com/y)." doesn't get captured as part of the URL itself.
    """
    raw = re.findall(r"https?://\S+", text or "")
    return sorted({re.sub(r'[.,;:!?)\]}"\']+$', "", u) for u in raw})


urls_cited = extract_urls(final_report_text)
print("URLs cited in final report:", urls_cited)

if len(urls_cited) >= 1:
    print()
    print("Attribution survived the relay: the coordinator's own final text")
    print("carries real source URLs, not just an unsourced summary.")
else:
    print()
    print("No URLs found in the final report -- that's the exact attribution")
    print("failure the case study near the end of this notebook reproduces on")
    print("purpose. If you're seeing it here instead, it's worth rereading the")
    print("prompts in Task 2 and above for anywhere attribution wasn't demanded.")


## 🛠️ Build Exercise — Task 6: Parallel Spawning

**Objective:** confirm the coordinator emits multiple `Agent` tool calls in
a **single** response for independent work, not one per turn.

**Why this matters:** this is the latency-awareness pattern from the concept
above, checked against real output instead of taken on faith. We ask for two
genuinely independent searches and then look at whether both `ToolUseBlock`s
landed in the same `AssistantMessage.content` list.


In [ ]:
PARALLEL_PROMPT = (
    "Using web_search_agent, research two independent subtopics of "
    "'renewable energy storage': (1) grid-scale battery storage, and "
    "(2) pumped-hydro storage. These are independent -- investigate both "
    "at once rather than one after the other. Report back a one-sentence "
    "finding for each, with its source URL."
)

max_agent_calls_in_one_turn = 0

async for message in query(prompt=PARALLEL_PROMPT, options=coordinator_options):
    if isinstance(message, AssistantMessage):
        agent_calls_this_turn = [
            b for b in message.content
            if isinstance(b, ToolUseBlock) and b.name in ("Agent", "Task")
        ]
        if agent_calls_this_turn:
            print(f"[one AssistantMessage] {len(agent_calls_this_turn)} Agent tool_use block(s):")
            for b in agent_calls_this_turn:
                print(f"    -> {b.input}")
            max_agent_calls_in_one_turn = max(max_agent_calls_in_one_turn, len(agent_calls_this_turn))

    elif isinstance(message, ResultMessage):
        print()
        print("Final:", message.result)

print()
if max_agent_calls_in_one_turn >= 2:
    print(f"Confirmed: {max_agent_calls_in_one_turn} Agent calls landed in a single response --")
    print("that's real parallel spawning, not sequential turns.")
else:
    print("The coordinator spread these across separate turns this run rather than")
    print("batching them into one response. Real model behavior varies call to call --")
    print("try rerunning, or make the 'investigate both at once' instruction more explicit.")


## ⚠️ Five Anti-Patterns to Avoid

| # | Anti-pattern | Why it fails | Fix |
|---|---|---|---|
| 1 | Assuming subagents inherit coordinator history or other subagents' outputs | Subagents have completely isolated context — nothing is inherited | Explicitly include everything needed in the subagent's own prompt |
| 2 | Blaming the synthesis step for missing citations | It can only cite what it was actually given | Fix the coordinator's context passing, not the synthesis prompt |
| 3 | Sequential invocation of independent subagent work | Adds pure latency with no benefit | Emit multiple `Agent` calls in one response (Task 6) |
| 4 | Confusing `fork_session` with `resume` | One branches, one continues — mixing them up loses or duplicates work | Fork to compare approaches; resume to continue the same one |
| 5 | Giving the synthesis step direct tool access to "re-verify" sources itself | Doesn't fix the root cause (missing metadata) — just adds complexity and cost | Fix what the coordinator passes, not what synthesis can do |

Each is written below as real, working code, then **commented out** — so you
can see the shape of the mistake without ever running it.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 1 -- Assuming a subagent inherits coordinator context
# ============================================================
# Commented out on purpose.
#
# bad_synthesis_agent = AgentDefinition(
#     description="Synthesizes a final report.",
#     prompt=(
#         "Synthesize a final report using the research findings from "
#         "earlier in this conversation."   # <-- there IS no "earlier" for this subagent
#     ),
#     tools=[],
# )
#
# Why it fails: a subagent's prompt is everything it knows. There is no
# "earlier in this conversation" from its point of view -- every invocation
# starts from zero, exactly as in Module 1.2. Findings must be typed
# directly into the prompt, not gestured at.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 2 -- Blaming the synthesis step for missing citations
# ============================================================
# Commented out on purpose.
#
# def diagnose_missing_citations_antipattern(report_text):
#     return "Fix: tell the coordinator's final synthesis to try harder to cite sources."
#
# Why it fails: if the coordinator only ever typed bare claim strings into
# its own final-response reasoning -- no source_url, no document_name -- no
# amount of "try harder" fixes a citation that was never possible to write.
# Check what was actually passed before touching the prompt asking for citations.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 3 -- Sequential invocation of independent work
# ============================================================
# Commented out on purpose. Compare this shape to Task 6's real, single-call
# parallel version above.
#
# async def run_sequential_antipattern(topic_a, topic_b):
#     result_a = None
#     async for message in query(prompt=f"Research {topic_a}", options=coordinator_options):
#         if isinstance(message, ResultMessage):
#             result_a = message.result
#     result_b = None
#     async for message in query(prompt=f"Research {topic_b}", options=coordinator_options):
#         if isinstance(message, ResultMessage):
#             result_b = message.result
#     return result_a, result_b
#
# Why it fails: topic_a and topic_b don't depend on each other, so there is
# no reason to wait for the entire first query() to finish (real, multi-turn,
# possibly tens of seconds) before even starting the second. Two full,
# separate coordinator sessions here is strictly slower than the one
# single-response version in Task 6, for identical results.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 4 -- Confusing fork_session with resume
# ============================================================
# Commented out on purpose.
#
# # WRONG: "forking" by just resuming twice -- both calls append to the SAME
# # session history, so they are not independent branches at all.
# options_a = ClaudeAgentOptions(resume=some_session_id)
# options_b = ClaudeAgentOptions(resume=some_session_id)
#
# Why it fails: resume alone always continues the named session -- calling it
# twice just means two continuations of one shared history, each seeing
# whatever the other already appended. To get two INDEPENDENT branches from
# the same starting point, fork_session=True has to be set explicitly
# alongside resume (see the verified reference pattern in the concept
# section above) -- fork is a modifier on resume, not a synonym for it.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 5 -- Giving synthesis direct tool access to "re-verify"
# ============================================================
# Commented out on purpose.
#
# over_complicated_prompt = (
#     "Synthesize a report from the findings above. You also have WebSearch "
#     "access -- feel free to re-run any query yourself to double check or "
#     "recover a source URL if one seems to be missing."
# )
#
# Why it fails: this papers over the actual bug (the coordinator stripped
# metadata on the way to synthesis) with extra tool access and extra cost,
# instead of fixing the one place the metadata actually got lost. It also
# reintroduces work the web_search_agent already scoped out -- now two
# different parts of the system can both search the web, for no clear reason.


## 📖 Case Study, Live: The Attribution Failure

The module's exact scenario: web search and document analysis both do solid,
well-sourced work — and the final report still comes out with no citations,
because the coordinator's own prompt never asked for attribution to be
preserved. Here, we reuse the *same* subagents from Task 2 (their prompts
already demand attribution) but give the **coordinator** a prompt that never
mentions sources at all — reproducing the bug at the one layer the module
says it actually lives in.

Another real, ~30–90 second call.


In [ ]:
BAD_COORDINATOR_PROMPT = (
    "Research 'solar panel efficiency trends' using your available "
    "subagents, then write a short summary of what you learn."
    # <-- Notice: nothing here asks for sources, URLs, or citations.
    #     Rule 2 from the context-passing concept never gets invoked.
)

bad_report_text = None

async for message in query(prompt=BAD_COORDINATOR_PROMPT, options=coordinator_options):
    if isinstance(message, ResultMessage):
        bad_report_text = message.result

urls_in_bad_report = extract_urls(bad_report_text)

print("=== Report from an attribution-blind coordinator prompt ===")
print(bad_report_text)
print()
print("URLs cited:", urls_in_bad_report)
print()

if len(urls_in_bad_report) < len(urls_cited):
    print(f"There it is: {len(urls_cited)} source(s) survived with the Task 4/5 prompt "
          f"that demanded attribution, vs. {len(urls_in_bad_report)} here, where the "
          f"coordinator's OWN prompt never asked for it -- even though these are the "
          f"exact same subagents, doing the same underlying research.")
else:
    print("This particular run still carried some attribution through -- real model")
    print("behavior varies. The mechanism is the same either way: attribution is only")
    print("as reliable as the coordinator's own prompt makes it, not a property of")
    print("the subagents themselves.")


def diagnose_missing_citations(report_text: str) -> str:
    """The correct diagnostic -- check what the COORDINATOR asked for, not
    whether the subagents did good work."""
    urls = extract_urls(report_text)
    if not urls:
        return (
            "Root cause: the coordinator's own prompt never asked for source "
            "attribution to be preserved through to the final synthesis. The "
            "subagents were never the problem -- the relay instruction was."
        )
    return f"{len(urls)} source(s) survived the relay."


print()
print(diagnose_missing_citations(bad_report_text))


## 🎓 Practice Scenario (from the module)

> A synthesis agent produces a report where several claims have no source
> attribution. The web-search subagent correctly returns results with URLs,
> titles, and snippets. The document-analysis subagent correctly returns
> analysis with page references. Both subagents are verified to be working
> properly. What is the most likely root cause?
>
> - A. Give the synthesis agent direct web-search access to re-run queries and verify sources itself
> - B. The coordinator passes content to the synthesis agent without structured metadata — source URLs, document names, and page numbers are not included
> - C. The synthesis agent's system prompt lacks explicit citation instructions
> - D. The web-search subagent returns results in a format the synthesis agent can't parse
>
> **Answer: B.** A adds complexity without fixing the root cause. C can't
> help — no prompt change lets an agent cite metadata it was never given. D
> is ruled out by the scenario itself: the web-search subagent is verified
> working correctly.


## 🏆 Key Takeaways for Exam Prep

1. **`Agent`/`Task` in `allowed_tools` is a hard, binary gate** — no
   subagent, however well-defined, can be spawned without it.
2. **Isolation still applies, at the API level too** — every subagent
   invocation starts from zero; nothing is inherited automatically.
3. **Attribution failures are a context-passing bug** — the fix is always in
   what the coordinator passes forward, never in the receiving agent's prompt.
4. **Structure content and metadata together** — a claim without its source
   is a claim that can never be cited downstream, by construction.
5. **Parallel spawning is a latency optimization** — independent subagent
   work belongs in one response, not serialized across turns.
6. **`fork_session` branches; `resume` continues** — and in the real SDK,
   fork is a modifier passed alongside resume, not a separate mechanism.


---

## 🎉 Quick-Fire Recap — See If It Stuck

You configured the real subagent-spawning primitive, watched attribution
survive a well-designed relay and then vanish from the exact same subagents
under a careless coordinator prompt, and (maybe) caught two searches landing
in one response. Try these from memory first.

**1. Your coordinator has two perfectly-defined `AgentDefinition`s in
`options.agents`, but subagent spawning still doesn't work. What's the first
thing to check?**
> 💡 Whether `"Agent"` (or `"Task"`) is actually in `allowed_tools`. It's a
> binary gate — a perfect `AgentDefinition` sitting unused doesn't help at all.

**2. A report comes back with zero citations, and every subagent involved is
verified to be working correctly. Where's the bug?**
> 💡 In the coordinator's own prompt or context-passing step — it never
> carried source metadata forward. Not the subagents; they can only cite what
> they were actually given.

**3. What's the real difference between the module's example field name
`system_prompt` and what you actually found in the installed SDK?**
> 💡 The real `AgentDefinition` field is `prompt`, not `system_prompt`. A
> good reminder that a study guide's pseudocode and a library's real, current
> API can quietly diverge — check the installed package when it matters.

**4. Two subtopics are completely independent. Your coordinator investigates
them one after another, across two turns. What's the fix, and why?**
> 💡 Emit both `Agent` tool calls in a single response instead. Sequential
> turns for independent work is pure added latency, with nothing gained.

**5. Someone wants two independent explorations of the same codebase
analysis. Should they use `resume` twice, or `fork_session`?**
> 💡 `fork_session=True` alongside `resume` — resuming twice just continues
> one shared history from two places, so neither exploration is actually
> independent of the other.

**6. In your own Task 4/5 run above, how many source URLs actually survived
into the final report — and in the case-study run right after, with a
prompt that never mentioned sources at all?**
> 💡 Whatever your two numbers were, the *gap* between them is the entire
> lesson: same subagents, same underlying research, different coordinator
> prompt — and attribution lived or died entirely on that one difference.

**7. A teammate suggests giving the synthesis step its own web-search access
"just in case a source goes missing." Good idea?**
> 💡 No — it hides the actual bug (missing metadata in what the coordinator
> passed) behind extra tool access and cost. Fix the relay, not the symptom.

---

### 🚀 Nice work.

Three modules into Domain 1: you've hand-built a single agentic loop, a
hand-rolled multi-agent coordinator, and now configured the real, built-in
subagent primitive that replaces the hand-rolling entirely in production.
Onward to **1.4 — Workflow Enforcement and Handoff**.
